In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown as md
import time as time

beep = lambda x: os.system("echo -n '\a';sleep 0.2;" * x)

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# Make graphs large enough to read easily
plt.clf()
plt.rcParams["figure.figsize"] = [10.0, 10.0*2/3]
matplotlib.rcParams["font.size"] = 20
None

In [ ]:
display(md("### load the code"))
%run ../gui/wja_caen_tcal
tbegin = time.time()

In [ ]:
f.load_drs_corrections()
print(list(f.tcal[0].__dict__.keys()))

In [ ]:
t0 = time.time()
f.do_triggered_readout(nevents=2000)
print(list(f.trigev[0].__dict__.keys()))
print(f"elapsed time {time.time()-t0:.1f}s")
beep(1)

In [ ]:
f.correct_triggered_readout()

In [ ]:
for ich in [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17]:
    for ev in range (1):
        plt.plot(f.trigev[ev].drsu[ich])

In [ ]:
hdf5_fnam = "caen_ch00vsch15.hdf5"

ns = SimpleNamespace()
ns.drsraw = np.array([e.drsraw for e in f.trigev], dtype=np.int16)
ns.drs = np.array([e.drsgc for e in f.trigev], dtype=np.float32)
ns.drsu = np.array([e.drsu for e in f.trigev], dtype=np.float32)
ns.traw = np.array([e.traw for e in f.trigev], dtype=np.float32)
ns.tcor = np.array([e.tcor for e in f.trigev], dtype=np.float32)
ns.drs_trig_cell = np.array([e.drs_trig_cell for e in f.trigev], dtype=np.int16)
ns.drs_tstamp = np.array([e.tstamp for e in f.trigev], dtype=np.int64)
ns.drs_cellgains = np.array([tc.cellgain for tc in f.tcal], dtype=np.float32)
ns.drs_cellwidth = np.array([tc.celldt for tc in f.tcal], dtype=np.float32)
ns.drs_peds = np.array([tc.cellpeds for tc in f.tcal], dtype=np.float32)
ns.drs_wiggle_shape = np.array([tc.wiggleshape for tc in f.tcal], dtype=np.float32)
ns.drs_mean_shape = np.array([tc.meanshape for tc in f.tcal], dtype=np.float32)

if "hf" in vars():
    hf.close()
    del hf

hf = h5py.File(hdf5_fnam, "w")

def cds(dsetname, comment):
    ds = hf.create_dataset(
        dsetname, 
        data=ns.__dict__[dsetname],
        shuffle=True,
        compression="gzip",
        compression_opts=1)
    ds.attrs["comment"] = comment

cds("drsraw", "raw uncorrected DRS samples [event][channel][sample]")
cds("drs", "DRS samples with pedestal and gain corrections applied")
cds("drsu", "DRS corrected samples, resampled to equal time intervals")
cds("traw", "nominal uncorrected DRS sample times")
cds("tcor", "DRS sample times, corrected via timing calibration")
cds("drs_trig_cell", "DRS trigger/stop cell ID [event][channel]")
cds("drs_tstamp", "CAEN board time stamp [event]")
cds("drs_cellgains", "voltage gain [channel][cell] from DRS calibration")
cds("drs_cellwidth", "cell width (ns) [channel][cell] from DRS timing calibration")
cds("drs_peds", "DRS pedestal [channel][cell] from DRS calibration")
cds("drs_wiggle_shape", "fitted 'wiggle' shape subtracted from each DRS waveform")
cds("drs_mean_shape", "mean waveform, in absence of signal, CAEN board artifact")
hf.close()
del hf
beep(1)

In [ ]:
iev = 800
e = f.trigev[iev]
print(e)
chnls = [0,1,2,3,8,9,10,16,17]  # these have input signals
if 0:
    # raw
    plt.clf()
    [plt.plot(e.traw[ichnl], e.drsraw[ichnl]) for ichnl in range(e.drsps.shape[0])]
    plt.grid()
    plt.xlabel("time [ns]")
    plt.ylabel("DRS ADC counts")
    plt.show()
    # pedestal-subtracted
    plt.clf()
    [plt.plot(e.traw[ichnl], e.drsps[ichnl]) for ichnl in range(e.drsps.shape[0])]
    plt.grid()
    plt.xlabel("time [ns]")
    plt.ylabel("DRS ADC counts")
    plt.show()
    # gain-corrected
    plt.clf()
    [plt.plot(e.traw[ichnl], e.drsgc[ichnl]) for ichnl in range(e.drsps.shape[0])]
    plt.grid()
    plt.xlabel("time [ns]")
    plt.ylabel("DRS ADC counts")
    plt.show()
# gain-corrected and tcal-corrected
plt.clf()
[plt.plot(e.tcor[ichnl], e.drsgc[ichnl]) for ichnl in chnls]
plt.grid()
plt.xlabel("tcal-corrected time [ns]")
plt.ylabel("DRS ADC counts")
plt.show()
None

In [ ]:
# Make CTR ntuple from triggered readout data stored in memory
ntuple = []
nplotted = 0
chnls = [0,1,2,3,8,9,10,16,17]  # these have input signals
for iev in range(len(f.trigev)):
    e = f.trigev[iev]
    nt = SimpleNamespace()
    for ich in chnls:
        t = e.tcor[ich]
        v = e.drsgc[ich] - e.drsgc[ich].mean()
        t = e.traw[ich]
        v = e.drsu[ich] - e.drsu[ich].mean()
        zct,zci = nbutil.zerocross1d(t, v, getIndices=True)
        cut = v[zci] < 0  # select rising edges
        zct = zct[cut]
        assert len(zct) == 1
        zc = zct[0]
        nt.__dict__["zc{}".format(ich)] = zc
    ntuple.append(nt)
dft = pd.DataFrame([_.__dict__ for _ in ntuple])

In [ ]:
# Make CTR ntuple from stored HDF5 file
tnow = time.localtime()
tnow = time.strftime("%Y_%m_%d_%H_%M_%S", tnow)
ifh = h5py.File("caen_%s.hdf5"%(tnow), "r")
drsu = ifh["drsu"][:]
traw = ifh["traw"][:]
ifh.close()
del ifh

ntuple = []
nplotted = 0
chnls = [0,1,2,3,8,9,10,16,17]  # these have input signals
for iev in range(drsu.shape[0]):
    nt = SimpleNamespace()
    for ich in chnls:
        t = traw[iev][ich]
        v = drsu[iev][ich] - drsu[iev][ich].mean()
        zct,zci = nbutil.zerocross1d(t, v, getIndices=True)
        cut = v[zci] < 0  # select rising edges
        zct = zct[cut]
        assert len(zct) == 1
        zc = zct[0]
        nt.__dict__["zc{}".format(ich)] = zc
    ntuple.append(nt)
dft = pd.DataFrame([_.__dict__ for _ in ntuple])


In [ ]:
# Report CTR values from ntuple
for i in range(3):
    expr = f"(zc17-zc{8+i})-(zc16-zc{0+i})"
    rms = dft.eval(expr).std()
    print(f"{i} : {expr} : rms = {1000*rms:.1f} ps")
for i in [0,1,2,8,9]:
    expr = f"zc{i}-zc{i+1}"
    rms = dft.eval(expr).std()
    print(f"{i} : {expr} : rms = {1000*rms:.1f} ps")

In [ ]:
dft.eval("zc16-zc17").std()

In [ ]:
print(iev, ich)
print(f.trigev[iev].tcor[ich])

In [ ]:
foobar = (t[1:] - t[:-1]) <= 0.0
print([i for i in range(len(foobar)) if foobar[i]])
print(t[907:912])

In [ ]:
idrs = 0
ichnl = 17
f.do_pedestal_bigkahuna(chnl=ichnl)

In [ ]:
t0 = time.time()
f.acquire_drs_cell_gains(nevents=1000, chnl=ichnl)
kh = f._kahuna
f.analyze_drs_cell_gains(chnl=ichnl)
print(f"elapsed time: {time.time()-t0:.1f}s")

In [ ]:
f.setup_tcal_acq(chnl=ichnl)
print(f"elapsed time since connecting to board: {time.time()-tbegin:.1f}s")

In [ ]:
t0 = time.time()
d = "../gui"
if d not in sys.path:
    sys.path.append(d)
import wja_caen_tcal as wct
from importlib import reload
reload(wct)
wct.do_tcal(f, chnl=ichnl)
print(f"elapsed time {time.time()-t0:.0f} seconds")

In [ ]:
iev = 10
k = (idrs,ichnl)
cal = kh.tcal_all[k]
tc = cal.cellid[iev]
cw = np.roll(cal.celldt, -(tc-1))
tcor = np.cumsum(cw) - cw[0]
traw = np.arange(len(cw)) * cw.mean()
v = cal.gaincorrected[iev]
#
inl = np.cumsum(cal.celldt) - cal.celldt.mean()*np.arange(len(cal.celldt))
plt.plot(np.arange(1024)[0::2], 1000*inl[0::2], '-')
plt.plot(np.arange(1024)[1::2], 1000*inl[1::2], '-')
plt.grid()
plt.xlabel(f"INL [ps] vs cell idrs={idrs} ichnl={ichnl}")
plt.show()
#
lo = 0
hi = -1
v = v[lo:hi]
tcor = tcor[lo:hi]
traw = traw[lo:hi]
t = traw
#
label = "uncorrected"
for _ in range(2):
    p0 = [v.mean(), (v.max()-v.min())/2, 10, 0]
    bounds = [
        (p0[0]-200, p0[0]+200),  # baseline
        (p0[1]*0.8, p0[1]*1.2),  # amplitude
        (p0[2]*0.9, p0[2]*1.1),  # period
        (-0.6*p0[2], +0.6*p0[2]) # time offset
    ]
    bounds = np.array(bounds).transpose()
    par,cov = scipy.optimize.curve_fit(
        kh.tcal_sine_func, t, v, p0=p0, bounds=bounds)
    print(p0)
    print(par)
    vfit = kh.tcal_sine_func(t, *par)
    resid = v-vfit
    rmsresid = resid.std()
    print(f"rms residual = {rmsresid}")
    plt.plot(t, v, 'o-')
    plt.plot(t, vfit, 'r-')
    #plt.plot(t, 1000*cw[:len(t)], 'go')
    plt.plot(t, 10*resid, 'g+')
    plt.xlabel(f"samples & sine fit vs t ({label}) idrs={idrs} ichnl={ichnl}")
    plt.axis([0,200,-1300,1300])
    plt.grid()
    plt.show()
    t = tcor
    label = "corrected"

In [ ]:
plt.plot(kh.meanshape); plt.plot(kh.wiggleshape)

In [ ]:
print(f"idrs={idrs} ichnl={ichnl}")
tco = list(kh.tcal_all.values())[0]
assert tco.idrs == idrs
assert tco.ichnl == ichnl
assert idrs == 0  # we'll just number channels from 0 to 15 on the CAEN board
print(kh.cellpeds.shape)
print(kh.cellgain.shape)
print(tco.celldt.shape)
print(kh.wiggleshape.shape)
print(kh.meanshape.shape)

In [ ]:
import h5py
hf = h5py.File("caen_tcal.hdf5", "a")
tstamp = time.strftime("%Y%m%d-%H%M")
hdf5_group_name = f"/tcal-{tstamp}-ch{ichnl:02d}"
print(hdf5_group_name)
hg = hf.create_group(hdf5_group_name)
hg.attrs["idrs"] = idrs
hg.attrs["ichnl"] = ichnl
hg.attrs["tstamp"] = tstamp
hg.create_dataset("cellpeds", data=kh.cellpeds)
hg.create_dataset("cellgain", data=kh.cellgain)
hg.create_dataset("celldt", data=tco.celldt)
hg.create_dataset("wiggleshape", data=kh.wiggleshape)
hg.create_dataset("meanshape", data=kh.meanshape)
hf.close()

In [ ]:
run export_to_html --timestamp